In [ ]:
# Libraries
import pandas as pd
import pickle
import numpy as np
from tqdm import tqdm

In [ ]:
data = pd.read_csv("../input/Data_API.csv", dtype=str)

data["Datetime_updated_seconds"]=data['Datetime_updated_seconds'].apply(pd.to_datetime)

data.head()

In [ ]:
from pandas import Timestamp
nft2txs={}

for group_id,transactions in tqdm(data.groupby("Unique_id_collection"),"Generating transactions"):

    nft=transactions["Unique_id_collection"].values[0]
    category=transactions["Category"].values[0]
    collection=transactions["Collection"].values[0]
    timestamps=[Timestamp(x) for x in transactions["Datetime_updated_seconds"].values]

    nft2txs[nft]={
        "txs":[{
            "price":price,
            "timestamp":timestamp,
            "time":time,
            "seller":seller,
            "buyer":buyer
        } for price,timestamp,time,seller,buyer in zip(
            transactions["Price_USD"].values,
            timestamps,
            transactions["Datetime_updated_seconds"].values,
            transactions["Seller_address"].values,
            transactions["Buyer_address"].values
            )],

            
        "category":category,
        "collection":collection,
    }



In [ ]:
pickle.dump(nft2txs,open("../input/nft2txs.pkl","wb"))

# Scraping time creation

In [ ]:
import requests
import json
import time
from pandas import Timestamp

def scrapeCollection(collection_name):
    url = f"https://api.opensea.io/api/v1/collection/{collection_name}"
    response = requests.get(url)
    time.sleep(0.2)
    return json.loads(response.text)
def toDate(date_str):
    t=Timestamp(date_str)
    return f'{t.year}/{t.month}/{t.day}'
coll2date={}

In [ ]:
for coll in tqdm(list(data["Collection"].unique())):
    try:
        scaped_data=scrapeCollection(coll.lower())
        date=toDate(scaped_data["collection"]["created_date"])
        coll2date[coll]=date
    except:
        coll2date[coll]=None
pickle.dump(coll2date,open("../input/coll2created.pkl","wb"))

In [ ]:
pickle.dump(coll2date,open("../input/coll2created.pkl","wb"))

In [ ]:
len([x for x in coll2date if coll2date[x] is not None])